In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
import matplotlib.pyplot as plt

In [ ]:
from IPython.display import display

TRAIN_PATH = "Devex_train.csv"
TEST_PATH = "Devex_test_questions.csv"

train_df = pd.read_csv(TRAIN_PATH, encoding="latin-1")
test_df = pd.read_csv(TEST_PATH, encoding="latin-1")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# snip display of the train and test datasets
print("\nTrain data (first 2 rows):")
display(train_df.head(2))

print("\nTest data (first 2 rows):")
display(test_df.head(2))

In [ ]:
# Parsing Labels
LABEL_COLS = [f"Label {i}" for i in range(1, 13)]


def extract_labels(row):
    """Return active SDG indicator strings for one row (ignore NA/NaN/empty)."""
    labels = []
    for col in LABEL_COLS:
        value = row[col]
        if pd.isna(value):
            continue
        value = str(value).strip()
        if value and value.upper() != "NA":
            labels.append(value)
    return labels


train_df["labels"] = train_df.apply(extract_labels, axis=1)
train_df["n_labels"] = train_df["labels"].apply(len)

# Build fixed label vocabulary from training set only
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(train_df["labels"])
label_names = mlb.classes_

print(f"Number of unique SDG 3 indicators: {len(label_names)}")
print(f"Label matrix shape (samples × labels): {Y.shape}")
print(f"Labels per document — min: {train_df['n_labels'].min()}, "
      f"max: {train_df['n_labels'].max()}, "
      f"mean: {train_df['n_labels'].mean():.2f}")

# Shared split settings for preprocessing ablation, baseline, and experiments 1–10
RANDOM_STATE = 42
TEST_SIZE = 0.2
y = Y

display(train_df[["Unique ID", "Type", "n_labels", "labels"]].head(3))


In [ ]:
# EDA visualizations

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1) Labels per document
train_df["n_labels"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0, 0], color="steelblue", edgecolor="black"
)
axes[0, 0].set_title("Labels per document")
axes[0, 0].set_xlabel("# indicators")
axes[0, 0].set_ylabel("# documents")

# 2) Document type distribution
train_df["Type"].value_counts().plot(
    kind="barh", ax=axes[0, 1], color="coral", edgecolor="black"
)
axes[0, 1].set_title("Document types (train)")

# 3) Top 10 most frequent indicators
label_counts = Y.sum(axis=0)
top_idx = np.argsort(label_counts)[::-1][:10]
top_labels = [
    label_names[i][:45] + "..." if len(label_names[i]) > 45 else label_names[i]
    for i in top_idx
]
axes[1, 0].barh(
    top_labels[::-1], label_counts[top_idx][::-1], color="seagreen", edgecolor="black"
)
axes[1, 0].set_title("Top 10 SDG 3 indicators (frequency)")
axes[1, 0].set_xlabel("# documents")

# 4) Text length (characters) before cleaning
train_df["text_len"] = train_df["Text"].astype(str).str.len()
test_df["text_len"] = test_df["Text"].astype(str).str.len()
axes[1, 1].hist(
    train_df["text_len"], bins=50, alpha=0.7, label="Train", color="steelblue"
)
axes[1, 1].hist(test_df["text_len"], bins=50, alpha=0.7, label="Test", color="orange")
axes[1, 1].set_title("Text length distribution (raw)")
axes[1, 1].set_xlabel("Characters")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("\nLabel prevalence (documents containing each indicator):")
prevalence = pd.Series(label_counts, index=label_names).sort_values(ascending=False)
display(prevalence.head(10).to_frame("count"))

In [ ]:
# Class imbalance: full label distribution + imbalance ratio
label_counts = Y.sum(axis=0)
prevalence = pd.Series(label_counts, index=label_names).sort_values(ascending=False)
max_count = prevalence.max()
min_count = prevalence.min()

imbalance_df = pd.DataFrame({
    "indicator": label_names,
    "count": label_counts,
    "pct_documents": (label_counts / len(train_df) * 100).round(2),
    "imbalance_ratio": (max_count / np.maximum(label_counts, 1)).round(1),
}).sort_values("count", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
short_all = imbalance_df["indicator"].str.replace(r"^(\d+\.\S+) - .*", r"\1", regex=True)
axes[0].barh(short_all.iloc[::-1], imbalance_df["count"].iloc[::-1], color="indianred", edgecolor="black")
axes[0].set_title("All 27 labels — document count")
axes[0].set_xlabel("# documents")
axes[0].axvline(50, color="gray", linestyle="--", alpha=0.7, label="50 docs (rare threshold)")
axes[0].legend()

rare = imbalance_df[imbalance_df["count"] < 50]
axes[1].bar(range(len(rare)), rare["count"], color="darkorange", edgecolor="black")
axes[1].set_xticks(range(len(rare)))
axes[1].set_xticklabels(
    rare["indicator"].str.replace(r"^(\d+\.\S+) - .*", r"\1", regex=True),
    rotation=45, ha="right", fontsize=8,
)
axes[1].set_title(f"Rare labels (< 50 docs): {len(rare)} indicators")
axes[1].set_ylabel("# documents")
plt.tight_layout()
plt.show()

print(f"Most frequent: {prevalence.index[0][:50]}... ({int(max_count)} docs)")
print(f"Least frequent: {prevalence.index[-1][:50]}... ({int(min_count)} docs)")
print(f"Imbalance ratio (max/min): {max_count/min_count:.1f}x")
display(imbalance_df.head(5))
display(imbalance_df.tail(5))